# 🏥 Laboratorio en Google Colab: Red Neuronal con PyTorch y Aprendizaje Federado (FedAvg)
### Diagnóstico Distribuido de Diabetes sin Compartir Datos Clínicos Privados
---
Este cuaderno interactivo replica celda por celda el motor de Deep Learning utilizado en el proyecto de telemedicina federada.
Aprenderás a:
1. Definir un Perceptrón Multicapa (`ModeloDiagnosticoNN`) con PyTorch.
2. Aplicar normalización Z-Score a variables clínicas.
3. Simular 3 hospitales independientes (`HospitalA`, `HospitalB`, `HospitalC`).
4. Implementar el algoritmo de agregación ponderada **Federated Averaging (FedAvg)**.
5. Trazar gráficas de convergencia y evaluar pacientes en tiempo real.

In [ ]:
# [CELDA 1] Importación de librerías esenciales y fijación de semillas
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🚀 Google Colab listo en dispositivo: {device}')
print(f'📦 PyTorch versión: {torch.__version__}')

In [ ]:
# [CELDA 2] Generación del dataset sintético con correlación médica
def generar_datos_clinicos(n_muestras=360):
    datos = []
    for _ in range(n_muestras):
        edad = random.uniform(22, 75)
        imc = random.uniform(19, 42)
        presion = random.uniform(95, 170)
        
        # Riesgo fisiológico correlacionado
        riesgo = (edad * 0.03) + (imc * 0.08) + (presion * 0.02)
        glucosa = random.uniform(70, 130) + (riesgo * 15)
        
        # Score logístico para diagnóstico de diabetes
        score = (glucosa - 110) * 0.04 + (imc - 27) * 0.1 + (edad - 45) * 0.03
        prob = 1.0 / (1.0 + np.exp(-score))
        diagnostico = 1.0 if (random.random() < prob or glucosa > 140) else 0.0
        
        datos.append([edad, imc, glucosa, presion, diagnostico])
    return np.array(datos, dtype=np.float32)

dataset_completo = generar_datos_clinicos(360)
print(f'✅ Total de pacientes generados: {len(dataset_completo)}')
print(f'📋 Primer paciente [Edad, IMC, Glucosa, Presión, Diagnóstico]:\n   {dataset_completo[0]}')

In [ ]:
# [CELDA 3] Segregación en 3 silos hospitalarios privados y test global
datos_hosp_a = dataset_completo[0:80]
datos_hosp_b = dataset_completo[80:160]
datos_hosp_c = dataset_completo[160:240]
datos_test_global = dataset_completo[240:360]

print(f'🏥 HospitalA (San Carlos):       {len(datos_hosp_a)} historias privadas')
print(f'🏥 HospitalB (Santa María):      {len(datos_hosp_b)} historias privadas')
print(f'🏥 HospitalC (Metropolitana):    {len(datos_hosp_c)} historias privadas')
print(f'🌐 Test Set Global de Validación:{len(datos_test_global)} historias')

In [ ]:
# [CELDA 4] Estandarización Z-Score y clase Dataset en PyTorch
MEDIAS = torch.tensor([48.0, 28.5, 125.0, 125.0], dtype=torch.float32)
DESVIACIONES = torch.tensor([15.0, 6.0, 35.0, 18.0], dtype=torch.float32)

def normalizar_features(tensor_x):
    return (tensor_x - MEDIAS) / (DESVIACIONES + 1e-7)

class DiabetesDataset(Dataset):
    def __init__(self, datos_numpy):
        x_raw = torch.tensor(datos_numpy[:, :4], dtype=torch.float32)
        self.x = normalizar_features(x_raw)
        self.y = torch.tensor(datos_numpy[:, 4:], dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

loader_a = DataLoader(DiabetesDataset(datos_hosp_a), batch_size=16, shuffle=True)
loader_b = DataLoader(DiabetesDataset(datos_hosp_b), batch_size=16, shuffle=True)
loader_c = DataLoader(DiabetesDataset(datos_hosp_c), batch_size=16, shuffle=True)
loader_test = DataLoader(DiabetesDataset(datos_test_global), batch_size=32, shuffle=False)

print('✅ DataLoaders inicializados correctamente.')

In [ ]:
# [CELDA 5] Definición de la Red Neuronal (Perceptrón Multicapa)
class ModeloDiagnosticoNN(nn.Module):
    def __init__(self):
        super(ModeloDiagnosticoNN, self).__init__()
        self.fc1 = nn.Linear(4, 32)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(32, 32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.relu1(self.fc1(x))
        out = self.relu2(self.fc2(out))
        out = self.sigmoid(self.fc3(out))
        return out

def serializar_pesos(modelo):
    return {k: v.detach().cpu().numpy().tolist() for k, v in modelo.state_dict().items()}

def deserializar_pesos(modelo, dict_pesos):
    state_dict = {k: torch.tensor(v, dtype=torch.float32) for k, v in dict_pesos.items()}
    modelo.load_state_dict(state_dict)

modelo_demo = ModeloDiagnosticoNN()
print(modelo_demo)
print(f'📊 Total de parámetros sinápticos: {sum(p.numel() for p in modelo_demo.parameters())}')

In [ ]:
# [CELDA 6] Función de entrenamiento privado local de un hospital
def entrenar_hospital_local(modelo_inicial, data_loader, epochs=3, lr=0.015):
    modelo_local = ModeloDiagnosticoNN()
    modelo_local.load_state_dict(modelo_inicial.state_dict())
    modelo_local.train()

    criterion = nn.BCELoss()
    optimizer = optim.Adam(modelo_local.parameters(), lr=lr)

    loss_final = 0.0
    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch_x, batch_y in data_loader:
            optimizer.zero_grad()
            pred = modelo_local(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        loss_final = epoch_loss / len(data_loader)

    modelo_local.eval()
    correctos = 0
    total = 0
    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            pred = modelo_local(batch_x)
            clase = (pred >= 0.5).float()
            correctos += (clase == batch_y).sum().item()
            total += len(batch_y)

    return serializar_pesos(modelo_local), loss_final, (correctos / total), total

In [ ]:
# [CELDA 7] Algoritmo FedAvg y evaluación global sobre el Test Set
def fed_avg(actualizaciones):
    total_muestras = sum(upd['num_muestras'] for upd in actualizaciones)
    pesos_agregados = {}
    claves = actualizaciones[0]['pesos'].keys()

    for key in claves:
        tensor_promedio = None
        for upd in actualizaciones:
            peso_ponderador = upd['num_muestras'] / total_muestras
            tensor_hosp = torch.tensor(upd['pesos'][key], dtype=torch.float32)
            if tensor_promedio is None:
                tensor_promedio = tensor_hosp * peso_ponderador
            else:
                tensor_promedio += tensor_hosp * peso_ponderador
        pesos_agregados[key] = tensor_promedio.detach().cpu().numpy().tolist()

    return pesos_agregados

def evaluar_modelo_global(modelo, test_loader):
    modelo.eval()
    criterion = nn.BCELoss()
    total_loss = 0.0
    correctos = 0
    total = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            pred = modelo(batch_x)
            total_loss += criterion(pred, batch_y).item()
            clase = (pred >= 0.5).float()
            correctos += (clase == batch_y).sum().item()
            total += len(batch_y)
    return (correctos / total), (total_loss / len(test_loader))

In [ ]:
# [CELDA 8] Ejecución de 5 rondas federadas sincrónicas
modelo_global = ModeloDiagnosticoNN()

historial_rondas = [0]
acc_0, loss_0 = evaluar_modelo_global(modelo_global, loader_test)
historial_acc = [acc_0 * 100]
historial_loss = [loss_0]
print(f'⭐ Ronda #0 (Inicial) -> Precisión Global: {acc_0*100:.2f}% | Pérdida: {loss_0:.4f}')

for r in range(1, 6):
    print(f'\n--- RONDA FEDERADA #{r} ---')
    w_a, loss_a, acc_a, n_a = entrenar_hospital_local(modelo_global, loader_a, epochs=3, lr=0.015)
    w_b, loss_b, acc_b, n_b = entrenar_hospital_local(modelo_global, loader_b, epochs=3, lr=0.015)
    w_c, loss_c, acc_c, n_c = entrenar_hospital_local(modelo_global, loader_c, epochs=3, lr=0.015)
    print(f'  🏥 HospA: Acc={acc_a*100:.1f}%, Loss={loss_a:.4f}')
    print(f'  🏥 HospB: Acc={acc_b*100:.1f}%, Loss={loss_b:.4f}')
    print(f'  🏥 HospC: Acc={acc_c*100:.1f}%, Loss={loss_c:.4f}')
    
    actualizaciones = [
        {'pesos': w_a, 'num_muestras': n_a},
        {'pesos': w_b, 'num_muestras': n_b},
        {'pesos': w_c, 'num_muestras': n_c}
    ]
    pesos_agregados = fed_avg(actualizaciones)
    deserializar_pesos(modelo_global, pesos_agregados)
    
    acc_glob, loss_glob = evaluar_modelo_global(modelo_global, loader_test)
    historial_rondas.append(r)
    historial_acc.append(acc_glob * 100)
    historial_loss.append(loss_glob)
    print(f'  🎯 MODELO GLOBAL RONDA #{r} -> Precisión: {acc_glob*100:.2f}% | Pérdida: {loss_glob:.4f}')

In [ ]:
# [CELDA 9] Visualización de las curvas de convergencia (Accuracy & Loss)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(historial_rondas, historial_acc, marker='o', color='#10b981', linewidth=2.5, label='Precisión Global (%)')
plt.title('Evolución de la Precisión Global (Test Set)', fontsize=12, fontweight='bold')
plt.xlabel('Ronda Federada')
plt.ylabel('Precisión (%)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(historial_rondas, historial_loss, marker='s', color='#f59e0b', linewidth=2.5, label='Pérdida Global (BCELoss)')
plt.title('Evolución de la Pérdida Global', fontsize=12, fontweight='bold')
plt.xlabel('Ronda Federada')
plt.ylabel('Pérdida (BCELoss)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# [CELDA 10] Diagnóstico interactivo de pacientes con el modelo global
def evaluar_paciente(edad, imc, glucosa, presion):
    modelo_global.eval()
    x_input = torch.tensor([[float(edad), float(imc), float(glucosa), float(presion)]], dtype=torch.float32)
    x_norm = normalizar_features(x_input)
    with torch.no_grad():
        prob = modelo_global(x_norm).item()
    
    diag = 'Positivo para Diabetes Mellitus' if prob >= 0.50 else 'Negativo / Bajo Riesgo'
    alerta = 'ALTA PRIORIDAD' if prob >= 0.70 else ('MODERADA' if prob >= 0.40 else 'BAJA')
    print(f'\n🩺 EVALUACIÓN CLÍNICA: Paciente (Edad={edad}, IMC={imc}, Glucosa={glucosa}, Presión={presion})')
    print(f'   Probabilidad Estimada: {prob*100:.2f}% | Diagnóstico: {diag} | Alerta: {alerta}')

# Evaluación de dos casos clínicos de prueba
evaluar_paciente(edad=58, imc=33.5, glucosa=165, presion=145)
evaluar_paciente(edad=24, imc=21.0, glucosa=85, presion=110)